In [ ]:
import json
import re
import os
import pandas as pd
from tqdm.std import tqdm
from lxml import html, etree

In [ ]:
annotation_data = "annotation/final_annotation_project.json"

In [ ]:
def load_annotations(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

In [ ]:
def sanitize_label(label: str) -> str:
    return re.sub(r'[^a-zA-Z0-9_-]', '', label).lower() or "mark"

In [ ]:
def normalize_xpath(raw_xpath: str) -> str:
    return raw_xpath.strip()

In [ ]:
def strip_text_suffix(xp: str):
    m = re.search(r'/text\(\)\[(\d+)\]$', xp)
    if m:
        base = xp[:m.start()]
        idx = int(m.group(1))
        return base, idx
    return xp, None

In [ ]:
def _pick_annotation(task_obj: dict, preferred_annotator: int):
    anns = task_obj.get("annotations", [])
    if not anns:
        raise ValueError("no annotations in this task")
    if len(anns) == 1:
        return anns[0]
    for a in anns:
        if a.get("completed_by") == preferred_annotator:
            return a
    return anns[0]


In [ ]:
def _get_text_slot(elem: etree._Element, text_idx: int):
    slots = []
    if elem.text is not None:
        slots.append((elem, "text", elem.text))
    for child in elem:
        if child.tail is not None:
            slots.append((child, "tail", child.tail))
    if 1 <= text_idx <= len(slots):
        return slots[text_idx - 1]
    return None, None, None

In [ ]:
def _find_elem_with_fallback(root, base_xpath: str):
    # 1. try original xpath
    try:
        elems = root.xpath(base_xpath)
    except Exception:
        elems = []

    if elems:
        return elems

    # 2. relaxed search
    relaxed = "//" + base_xpath.lstrip("/")
    try:
        elems = root.xpath(relaxed)
    except Exception:
        elems = []

    if elems:
        return elems

    # 3. special fallback: patterns like /font[19]
    m = re.fullmatch(r'/([a-zA-Z0-9]+)\[(\d+)\]', base_xpath)
    if m:
        tag = m.group(1)
        idx = int(m.group(2))
        all_of_tag = root.xpath(f"//{tag}")
        if 1 <= idx <= len(all_of_tag):
            return [all_of_tag[idx - 1]]

    return []

In [ ]:
def find_textnode_by_literal(root: etree._Element, literal: str):
    """
    Search the entire tree for a text or tail containing the literal.
    Only return when there is exactly one match; return None if zero or multiple matches are found.
    """
    literal = literal or ""
    if not literal:
        return None, None  # elem, kind

    matches = []
    # traverse all nodes
    for elem in root.iter():
        # check elem.text
        if elem.text and literal in elem.text:
            matches.append((elem, "text"))
        # check tail of each child
        for ch in elem:
            if ch.tail and literal in ch.tail:
                matches.append((ch, "tail"))

    if len(matches) == 1:
        return matches[0]  # (owner, kind)
    else:
        return None, None

In [ ]:
def recover_offsets_from_literal(current_text: str, literal: str):
    """
    Use the annotated literal text to search again within current_text.
    If found, return (start, end); if not found, return (None, None).
    """
    if not current_text or not literal:
        return None, None
    pos = current_text.find(literal)
    if pos == -1:
        # try again after light cleaning (remove commas, dollar signs)
        cleaned_current = current_text.replace(",", "").replace("$", "")
        cleaned_literal = literal.replace(",", "").replace("$", "")
        pos = cleaned_current.find(cleaned_literal)
        if pos == -1:
            return None, None
        # the cleaned index cannot be accurately mapped back to original text, so give up
        return None, None
    return pos, pos + len(literal)

In [ ]:
def highlight_task_html(task_obj: dict, preferred_annotator: int = 10):
    raw_html = task_obj["data"]["matched_html"]
    root = html.fromstring(raw_html)

    total_found = 0
    number_count = 0
    date_count = 0

    ann = _pick_annotation(task_obj, preferred_annotator)
    used_annotator = ann.get("completed_by")

    body_nodes = root.xpath("//body")
    body = body_nodes[0] if body_nodes else root

    for res in ann.get("result", []):
        if res.get("type") != "labels":
            continue
        value = res.get("value") or {}
        raw_xpath = (value.get("start") or "").strip()
        labels = value.get("labels") or []
        literal_text = value.get("text") or ""   # the actual annotated string
        if not raw_xpath or not labels:
            continue

        label_name = sanitize_label(labels[0])
        start_off = value.get("startOffset")
        end_off = value.get("endOffset")

        # ① body level
        if raw_xpath.startswith("/text()["):
            original_text = body.text or ""
            # first try offsets from annotation
            ok_range = (
                isinstance(start_off, int) and isinstance(end_off, int)
                and 0 <= start_off <= end_off <= len(original_text)
            )
            if not ok_range and literal_text:
                # fallback: recover offsets using literal text
                rec_s, rec_e = recover_offsets_from_literal(original_text, literal_text)
                if rec_s is not None:
                    start_off, end_off = rec_s, rec_e
                    ok_range = True

            if not ok_range:
                print(f"[WARN] page-level text offsets out of range: {start_off}, {end_off}, len={len(original_text)}")
                continue

            before = original_text[:start_off]
            middle = original_text[start_off:end_off]
            after = original_text[end_off:]

            new_tag = etree.Element(label_name)
            new_tag.text = middle

            old_children = list(body)
            body.text = before
            body.insert(0, new_tag)
            new_tag.tail = after
            for ch in old_children:
                body.append(ch)

            total_found += 1
            if labels[0] == "Number":
                number_count += 1
            elif labels[0] == "Date":
                date_count += 1

            print(f"✔ Wrapped [{label_name}] -> '{middle}' (body level)")
            continue

        # ② element-level
        norm_xpath = normalize_xpath(raw_xpath)
        base_xpath, text_idx = strip_text_suffix(norm_xpath)

        elems = _find_elem_with_fallback(root, base_xpath)
        if not elems:
            # fallback: use literal text to find a text node in the tree
            owner, owner_kind = find_textnode_by_literal(root, literal_text)
            if owner is None:
                print(f"[WARN] XPath not found even after fallback: {norm_xpath}")
                continue
            # manually construct the current text slot
            current_text = owner.text if owner_kind == "text" else owner.tail
            # try recovering offsets using literal
            rec_s, rec_e = recover_offsets_from_literal(current_text, literal_text)
            if rec_s is None:
                print(f"[WARN] found text node by literal but cannot recover offsets: '{literal_text}'")
                continue

            before = current_text[:rec_s]
            middle = current_text[rec_s:rec_e]
            after = current_text[rec_e:]

            new_tag = etree.Element(label_name)
            new_tag.text = middle

            if owner_kind == "text":
                parent = owner
                old_children = list(parent)
                parent.text = before
                parent.insert(0, new_tag)
                new_tag.tail = after
                for ch in old_children:
                    parent.append(ch)
            else:
                parent = owner.getparent()
                owner.tail = before
                idx_in_parent = list(parent).index(owner)
                parent.insert(idx_in_parent + 1, new_tag)
                new_tag.tail = after

            total_found += 1
            if labels[0] == "Number":
                number_count += 1
            elif labels[0] == "Date":
                date_count += 1

            print(f"✔ Wrapped [{label_name}] -> '{middle}' (found by literal)")
            continue

        # normal: element exists
        elem = elems[0]

        # get text slot
        if text_idx is None:
            current_text = elem.text or ""
            owner = elem
            owner_kind = "text"
        else:
            owner, owner_kind, current_text = _get_text_slot(elem, text_idx)
            if owner is None:
                print(f"[WARN] text()[{text_idx}] not found under {base_xpath}")
                continue

        # first try offsets from annotation
        ok_range = (
            isinstance(start_off, int) and isinstance(end_off, int)
            and 0 <= start_off <= end_off <= len(current_text)
        )

        if not ok_range and literal_text:
            # fallback: recover offset by literal text
            rec_s, rec_e = recover_offsets_from_literal(current_text, literal_text)
            if rec_s is not None:
                start_off, end_off = rec_s, rec_e
                ok_range = True

        if not ok_range:
            print(f"[WARN] offsets out of range: {start_off}, {end_off}, len={len(current_text)}")
            continue

        before = current_text[:start_off]
        middle = current_text[start_off:end_off]
        after = current_text[end_off:]

        new_tag = etree.Element(label_name)
        new_tag.text = middle

        if owner_kind == "text":
            parent = owner
            old_children = list(parent)
            parent.text = before
            parent.insert(0, new_tag)
            new_tag.tail = after
            for ch in old_children:
                parent.append(ch)
        else:
            parent = owner.getparent()
            owner.tail = before
            idx_in_parent = list(parent).index(owner)
            parent.insert(idx_in_parent + 1, new_tag)
            new_tag.tail = after

        total_found += 1
        if labels[0] == "Number":
            number_count += 1
        elif labels[0] == "Date":
            date_count += 1

        print(f"✔ Wrapped [{label_name}] -> '{middle}'")

    print(f"\n✅ Total entities wrapped (by annotator {used_annotator}): {total_found}")
    print(f"   Number: {number_count}, Date: {date_count}")

    updated_html = etree.tostring(root, encoding="unicode")
    return updated_html, total_found, number_count, date_count


In [ ]:
def save_html_to_txt(updated_html: str, output_path: str, mode: str = "w", encoding: str = "utf-8") -> None:

    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    with open(output_path, mode, encoding=encoding) as f:
        f.write(updated_html)

    print(f"✅ Saved highlighted HTML to: {output_path}")

In [ ]:
data = load_annotations(annotation_data)

In [ ]:
# data[104]

In [ ]:
annotators_map = {
    10: "A",
    11: "B",
    4: "C",
    13: "D"
}

In [ ]:
# updated_html, total_entity, number_count, date_count = highlight_task_html(data[1],preferred_annotator=4)

In [ ]:
groud_truth_folder = "./gold_annotation_html"

total_entities = 0
number_counts = 0
date_counts = 0
for i in tqdm(range(len(data))):
    output_path = os.path.join(groud_truth_folder, f"gold_{i}.txt")
    updated_html, total_entity, number_count, date_count = highlight_task_html(data[i],preferred_annotator=4)
    total_entities += total_entity
    number_counts += number_count
    date_counts += date_count
    save_html_to_txt(updated_html, output_path)

In [ ]:
total_entities

In [ ]:
number_counts

In [ ]:
date_counts